In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, KFold, cross_validate
from sklearn.metrics import make_scorer, r2_score, mean_squared_error, mean_absolute_error

from sklearn.ensemble import RandomForestRegressor
from scipy.stats import randint, uniform

import os
import joblib

In [8]:
# 1- xgboost
# 2- adaboost
# 3- gradient boosting
# 4- catboost
# 5- lightboost
# 6- random forest
# 7- stacking 01
# 8- stacking 02

In [9]:
df = pd.read_csv("../data/preprocessed_dataset.csv")

df.head(2)

,FA (kg/m3),GGBFS (kg/m3),Coarse aggregate (kg/m3),Fine aggregate (kg/m3),Na2SiO3,NaOH,Sio2 (Dry),Concentration (M) NaOH,Superplasticizer,Total water,Initial curing time (day),Initial curing temp (C),Initial curing rest time (day),Final curing temp (C),Water_to_Binder,Water_to_Activator,SP_ratio,total_material_mass,Total_CO2_kg,Compressive_strength_MPa
0,0.000000,0.069565,-0.127973,-0.495765,0.700467,0.741549,0.756779,0.0,0.095238,0.651360,0.0,0.0,0.0,0.0,0.629716,-0.436364,0.107509,0.885142,512.210033,46.57
1,0.140762,0.162319,-0.465568,0.198785,0.900600,0.950927,0.972882,0.0,0.142857,0.849623,0.0,0.0,0.0,0.0,0.629716,-0.436364,0.159822,0.948367,534.887466,46.01


In [10]:
labels = df[['Total_CO2_kg', 'Compressive_strength_MPa']]
df.drop(columns=['Total_CO2_kg', 'Compressive_strength_MPa'], inplace=True)

In [11]:
X_train, X_test, y_train, y_test = train_test_split(df, labels, random_state=42, test_size=0.20)

# Random Forest

In [12]:
rf = RandomForestRegressor(
    random_state=42,
    n_jobs=1
)

param_dist = {
    "n_estimators": randint(200, 1500),

    "max_depth": [None, 5, 10, 20, 30],
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(1, 20),

    "max_features": ["sqrt", "log2", None],

    "bootstrap": [True, False],
}

cv = KFold(n_splits=10, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=100,
    scoring="r2",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=4,
    error_score="raise"
)

random_search.fit(X_train, y_train)

print("Best Hyperparameters:", random_search.best_params_)
print("Best R2:", random_search.best_score_)


Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Best Hyperparameters: {'bootstrap': True, 'max_depth': 20, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 4, 'n_estimators': 327}
Best R2: 0.876401619546616


In [13]:
best = random_search.best_params_

rf = RandomForestRegressor(
    random_state=42,
    n_jobs=1
)

grid = {
    "n_estimators": sorted(set([
        max(100, best["n_estimators"] - 200),
        best["n_estimators"],
        best["n_estimators"] + 200,
    ])),

    "max_depth": [
        best["max_depth"],
        (best["max_depth"] + 5) if best["max_depth"] is not None else 10,
        (best["max_depth"] - 5) if (best["max_depth"] is not None and best["max_depth"] > 5) else None,
    ],

    "min_samples_split": sorted(set([
        max(2, best["min_samples_split"] - 2),
        best["min_samples_split"],
        best["min_samples_split"] + 2,
    ])),

    "min_samples_leaf": sorted(set([
        max(1, best["min_samples_leaf"] - 2),
        best["min_samples_leaf"],
        best["min_samples_leaf"] + 2,
    ])),

    "max_features": list(dict.fromkeys([
        best["max_features"],
        "sqrt" if best["max_features"] != "sqrt" else "log2",
    ])),

    "bootstrap": [best["bootstrap"]],
}

cv = KFold(n_splits=10, shuffle=True, random_state=42)

n_candidates = 1
for v in grid.values():
    n_candidates *= len(v)
print("Grid candidates:", n_candidates, "| Total fits:", n_candidates * cv.get_n_splits())

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=grid,
    scoring="r2",
    cv=cv,
    verbose=2,
    n_jobs=4,
    error_score="raise"
)

grid_search.fit(X_train, y_train)

print("Best params (Grid):", grid_search.best_params_)
print("Best R2 (Grid):", grid_search.best_score_)


Grid candidates: 108 | Total fits: 1080
Fitting 10 folds for each of 108 candidates, totalling 1080 fits
Best params (Grid): {'bootstrap': True, 'max_depth': 20, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 327}
Best R2 (Grid): 0.8889137337121215
